In [1]:
import spacy
import os
import json
import pandas as pd
import re
import requests
from periodictable import elements
from pathlib import Path
from pypdf import PdfReader as pr
#!pip install nltk
from nltk.tokenize import sent_tokenize
import pubchempy as pcp
import time
from biocyc import biocyc
import urllib.parse
from xml.etree import ElementTree as ET
import random

import warnings
warnings.filterwarnings("ignore")

In [2]:
#os.chdir('..\\dissertation DLC content')
#linker = Ontology('OntoBiotope_BioNLP-OST-2019.obo')

path_to_scispacy_model = os.path.normpath('C:\\Users\\jace\\Documents\\assignments\\dissertation DLC content\\en_ner_bionlp13cg_md-0.5.4\\en_ner_bionlp13cg_md\\en_ner_bionlp13cg_md-0.5.4')
path_to_custom_model = os.path.normpath('C:\\Users\\jace\\Documents\\assignments\\dissertation DLC content\\custom_ner')

scispacy_model = spacy.load(path_to_custom_model)
#microbial_chemical_model = spacy.load(path_to_scispacy_model)

# <font size=4>hidden</font>

In [3]:
%%script False

print(scispacy_model.pipe_names)

#linker = pyobo.from_obo_path('..\\dissertation DLC content\\chebi_lite.obo', version='1.2')
kb = KnowledgeBase('..\\dissertation DLC content\\chebi_lite.json')

linker = scispacy_model.add_pipe('entity_linker', after='ner')
linker.set_kb(kb)

onto = pyobo.fr('chebi_core')

#update this here to try and download a more relevant knowledge base
""" !python -m spacy_entity_linker "download_knowledge_base"

from spacy.kb import InMemoryLookupKB

def create_kb(vocab):
    kb = InMemoryLookupKB(vocab, entity_vector_length=128)
    return kb

linker = scispacy_model.add_pipe('entityLinker', after='ner') """

Couldn't find program: 'False'


# <font size=4>not hidden</font>

In [4]:
with open('list_files\\utilizations_list.json', 'r', encoding='utf-8') as f:
    utilizations = list(json.load(f))

basic_utilizations = ['fermentation', 'hydrolyze', 'hydrolyse', 'reduce', 'reduces', 'reduced', 'produce', 'production', 'produces', 'consume', 'consumes', 'consumption', 'consuming', 'producing', 'grow', 'growth', 'oxidise', 'oxidize', 'consumed', 'produced', 'convert', 'converts', 'converting', 'converted', 'conversion']

print('|'.join(basic_utilizations))

util_mappings = {
    'production': 'PRODUCES',
    'produce': 'PRODUCES',
    'produces': 'PRODUCES'
}
    
custom_labels = ['MICROBE_NAME', 'SIMPLE_CHEMICAL', 'CHEMICAL']

with open('list_files\\genus_names.json', 'r', encoding='utf-8') as f:
    genera_unfiltered = list(json.load(f))
    
with open('list_files\\food_list.json', 'r', encoding='utf-8') as f:
    food_list = list(json.load(f))
    
genus_names = []

for gen in genera_unfiltered:
    if 'Candidatus' in gen:
        gen_split = gen.split(' ')
        genus_names.append(gen_split[1])
    else:
        genus_names.append(gen)
        
with open('list_files\\microbes_genus_species.json', 'r', encoding='utf-8') as f:
    microbes_genus_species = list(json.load(f))
    
not_chemicals = ['flavour', 'flavor', 'flavours', 'flavors', 'carbohydrate', 'carbohydrates', 'vitamin', 'vitamins', 'aroma', 'aromas', 'flavonoid', 'flavonoids', 'co2', 'o2', 'liquid'] + genus_names + food_list
print(len(not_chemicals))
    
g_species = []
genus_species = []
for gs in microbes_genus_species:
    genus, species = gs.split(' ')
    g_species.append(str(genus[0] + '. ' + species))
    genus_species.append((genus, species))
    
print(len(g_species), len(set(g_species)))
    
with open('list_files\\species_names.json', 'r', encoding='utf-8') as f:
    species_names = list(set(json.load(f)))
    
with open('list_files\\basionym_realname_mapping.json', 'r', encoding='utf-8') as f:
    basionym_realname_mapping = dict(json.load(f))

with open('list_files\\pcp_metabolite_production_mapping.json', 'r', encoding='utf-8') as f:
    pcp_metabolite_production_mapping = dict(json.load(f))

with open('list_files\\pcp_metabolite_utilisation_mapping.json', 'r', encoding='utf-8') as f:
    pcp_metabolite_utilisation_mapping = dict(json.load(f))

pcp_mappings = {
    'production': pcp_metabolite_production_mapping,
    'utilisation': pcp_metabolite_utilisation_mapping
}

with open('list_files\\metabolite_production.json', 'r', encoding='utf-8') as f:
    metabolite_production = dict(json.load(f))

with open('list_files\\metabolite_utilisation.json', 'r', encoding='utf-8') as f:
    metabolite_utilisation = dict(json.load(f))

with open('list_files\\noresolve_chemicals_mapping.json', 'r', encoding='utf-8') as f:
    noresolve_chemicals_mapping = dict(json.load(f))

with open('list_files\\biocyc_db_mappings.json', 'r', encoding='utf-8') as f:
    biocyc_db_mappings = dict(json.load(f))

fermentation|hydrolyze|hydrolyse|reduce|reduces|reduced|produce|production|produces|consume|consumes|consumption|consuming|producing|grow|growth|oxidise|oxidize|consumed|produced|convert|converts|converting|converted|conversion
14934
24692 21518


Fixed mappings

In [14]:
microbe_shortforms_known_mappings = {
    'S. cerevisiae': 'Saccharomyces cerevisiae',
    'A. niger': 'Aspergillus niger',
    'S. fragilis': 'Saccharomyces fragilis',
    'L. hilgardii': 'Lentilactobacillus hilgardii'
}

chemical_resolved_unchanged = ['glucose', 'fructose', 'lactose']

numbers_map = {
    0: 'zero',
    1: 'one',
    2: 'two',
    3: 'three',
    4: 'four',
    5: 'five',
    6: 'six',
    7: 'seven',
    8: 'eight',
    9: 'nine'
}

fixed_chemical_resolutions = {
    'l-lactic acid': 'lactic acid',
    'l-lactate': 'lactic acid',
    'lactate': 'lactic acid',
    'l-malic acid': 'malic acid',
    'l-malate': 'malic acid',
    '(s)-lactate': 'lactic acid'
}

def preprocess_label(label):
    if label[0].isnumeric():
        label = label.replace(label[0], numbers_map[int(label[0])])
    replace_chars = ["-", "(", ")", " ", ".", ",", "\'", "/", "\"", ";", "&", ":", ";", "|", "%", "’", "?", "!", "«", "+", "（", "）", "？", "，"]
    for char in replace_chars:
        label = label.replace(char, '')
    return label

def preprocess_name(name):
    new_name = name
    name.replace('(', '').replace(')', '')
    new_name = new_name.strip()
    return new_name

def process_html(text):
    html_tags = re.findall('</?[A-Za-z]+>', text)

    for tag in html_tags:
        text = text.replace(tag, '')

    return text

In [ ]:
with open('list_files\\paper_number.json', 'r') as f:
    paper_no = int(json.load(f))

with open("list_files\\entity_label_map.json", 'r') as f:
    entity_label_map = dict(json.load(f))

with open('list_files\\reaction_enzyme_mapping.json', 'r', encoding='utf-8') as f:
    reaction_enzyme_mapping = dict(json.load(f))

with open('list_files\\biocyc_common_name_mapping.json', 'r', encoding='utf-8') as f:
    biocyc_common_name_mapping = dict(json.load(f))

#tuple of (microbe, consume/produce, chemical) or (chemical, become, chemical), and whether its true or false
with open('list_files\\saved_relations.json', 'r', encoding='utf-8') as f:
    saved_relations = set(json.load(f))

email = 'lp2095@bath.ac.uk'

with open('..\\dissertation DLC content\\go away\\passwords.json', 'r') as f:
    passwords = json.load(f)
    password = passwords.get('metacyc_pw')

with open('list_files\\pcp_chemical_mapping.json', 'r', encoding='utf-8') as f:
    chemical_mapping = dict(json.load(f))

session = requests.Session()

session.post('https://websvc.biocyc.org/credentials/login/', data={'email':email, 'password':password}, headers={'User-Agent': 'A University of Bath student. I am doing a dissertation project in Natural Language Processing, where I identify named entities such as microbes and chemicals in academic papers, and the relations between them. I am using the database for the relation extraction element, and whether a microbe consumes or produces a particular chemical.'})

for ind, file in enumerate(os.listdir("..\\dissertation DLC content\\fermentation_papers_preprocessed")[paper_no:paper_no+3]):
    microbe_shortform_mappings = {}
    print(ind)
    #keep track of longterm names of all microbes seen so far
    filename = os.fsdecode(file)
    if filename.endswith('json'):
        with open(f"..\\dissertation DLC content\\fermentation_papers_preprocessed\\{filename}", 'r', encoding='utf-8') as f:
            text = f.read()
    elif filename.endswith('pdf'):
        try:
            print('pdf!')
            path = Path(os.getcwd() + "\\fermentation_papers\\" + filename)
            reader = pr(path)
            text = ""
            for page in reader.pages:
                text = text + page.extract_text()
        except:
            continue
    else:
        continue
    
    def resolve_basionym(name):
        resolved_name = name
        basionym_mapping_name = basionym_realname_mapping.get(name)
        if basionym_mapping_name:
            resolved_name = basionym_mapping_name
        else:
            results = requests.get(f"https://api.gbif.org/v1/species/search?q={str(resolved_name.replace(' ', '+'))}").json().get('results')
            if results:
                gbif_name = results[0].get('species')
                if gbif_name:
                    resolved_name = gbif_name
        return resolved_name
    
    #do a first pass of the paper to identify microbe shortforms
    #TODO: dont process the whole paper or else my pc crashes, sentencize it first
    all_paper_ents = scispacy_model(text)
    all_mic_ents = [(mic.label_, mic.text, mic.start_char, mic.end_char) for mic in all_paper_ents.ents if mic.label_ == 'MICROBE_NAME']
    all_mic_shapes_raw = [mic.shape_ for mic in all_paper_ents]
    mic_shapes = [(name, ' '.join([msr[:4] for msr in all_mic_shapes_raw[s:e]]), sc, ec) for s, e, name, sc, ec in [(m.start, m.end, m.text, m.start_char, m.end_char) for m in all_paper_ents.ents if m.label_ == 'MICROBE_NAME']]
    for name, shape, s, e in mic_shapes:
        if shape == 'Xxxx xxxx':
            genus, species = name.split(' ')
            shortform_name = genus[0] + '. ' + species
            if not microbe_shortform_mappings.get(shortform_name):
                #normalize microbe name before adding
                #TODO: find alternative name, rather than just 'species'
                resolved_basionym = resolve_basionym(name)
                if resolve_basionym:
                    microbe_shortform_mappings[shortform_name] = resolved_basionym

        elif shape == 'X. xxxx' or shape == 'Xx. xxxx' or shape == 'Xxx. xxxx':
            if not microbe_shortform_mappings.get(name):
                #add that entry in to indicate we have seen the shortform before the longform, in case the shortform doesnt get seen again, sometimes the microbe names are weird
                microbe_shortform_mappings[name] = None

    del all_paper_ents
                
    #keep track of previous sentence and entities for which to merge with, if there are not enough to form relations with
    merge_sentence = None
    last_sent_length = 0
    merge_ents_resolved = []
    
    sentences = sent_tokenize(text)
    for sent in sentences:
        all_ents_raw = scispacy_model(sent)
        mic_ents_unfiltered = [(mic.label_, mic.text, mic.start_char, mic.end_char) for mic in all_ents_raw.ents if mic.label_ == 'MICROBE_NAME']

        #these are filtered but unresolved, so we keep the original, un-normalized names of the microbes, but they are filtered. this is done to remove people's names which
        #are misidentified as microbes
        mic_ents_filtered = []
        mic_shapes_raw = [mic.shape_ for mic in all_ents_raw]
        mic_shapes = [(name, ' '.join([msr[:4] for msr in mic_shapes_raw[s:e]]), sc, ec) for s, e, name, sc, ec in [(m.start, m.end, m.text, m.start_char, m.end_char) for m in all_ents_raw.ents if m.label_ == 'MICROBE_NAME']]
        mic_ents_resolved = []
        
        for name, shape, s, e in mic_shapes:
            if shape == 'X. xxxx' or shape == 'Xx. xxxx' or shape == 'Xxx. xxxx':
                alternate_name = None
                if shape == 'Xxx. xxxx':
                    print("weird name:", name)
                #check the map to see if theres anything in there
                shortform_match_name = microbe_shortform_mappings.get(name)
                #add the resolved name to the array
                if shortform_match_name:
                    resolved_name = resolve_basionym(shortform_match_name[0] + " " + shortform_match_name[1])
                    alternate_name = resolved_name
                    mic_ents_resolved.append(('MICROBE_NAME', resolved_name, s, e))
                else:
                    #try with the known mappings (obviously not gonna do them all, just the most common ones)
                    known_name = microbe_shortforms_known_mappings.get(name)
                    if known_name:
                        alternate_name = known_name
                        microbe_shortform_mappings[name] = known_name
                        mic_ents_resolved.append(('MICROBE_NAME', known_name, s, e))
                    else:
                        #perhaps match against the list of genera and species to see which is the most probable? like start with the first letter and figure shit out from there
                        #like check the species name and check it against the full names, and see if theres any genera which match up
                        #but what if there are multiple candidates?
                        
                        full_genus, full_species = name.split(' ') #genus is first letter, species is 'xxxx' bit after
                        g_species = full_genus[0] + '. ' + full_species
                        #originally dont match with genus to check for potential name changes
                        matched_species = [(gen, spec) for gen, spec in genus_species if spec == full_species]
                        
                        matched_name = None
                        
                        if len(matched_species) == 1:
                            matched_name = matched_species[0]
                        elif len(matched_species) == 2:
                            gen1, spec1 = matched_species[0]
                            resolved_basionym_1 = resolve_basionym(str(gen1 + '+' + spec1))
                            
                            gen2, spec2 = matched_species[1]
                            resolved_basionym_2 = resolve_basionym(str(gen2 + '+' + spec2))
                            
                            if resolved_basionym_1 == resolved_basionym_2:
                                matched_name = (gen1, spec1) #found match!
                                microbe_shortform_mappings[name] = gen1 + '+' + spec1
                            else:
                                #TODO: figure something out idk
                                pass
                        else:
                            #match with genus name rather than just species
                            matched_genus = [(gen, spec) for gen, spec in matched_species if gen[0] == full_genus[0]]
                            if len(set(matched_genus)) == 1:
                                matched_name = matched_genus[0] #found match!
                            else:
                                #figure something out idk, maybe the same thing as above
                                pass
                                
                        alternate_name = matched_name
                               
                        #resolve name and add mapping 
                        if matched_name:
                            resolved_basionym = resolve_basionym(' '.join(matched_name))
                            mic_ents_resolved.append(('MICROBE_NAME', str(resolved_basionym), s, e))
                            #only add the original name to the shortform mapping, not the resolved names
                            microbe_shortform_mappings[name] = matched_name
                            
                        #TODO: consider cases such as 'Lpb. plantarum' - IN PROGRESS
                        
                #sometimes an entity is flagged as a microbe when it is actually not. however, i want to keep the original microbe names for the relation extraction part-
                #the llm will see the context sentence and if we give it the resolved entities it will not know how to match them.
                #so only add to the filtered list if an alternate, resolved name exists.
                if alternate_name:
                    mic_ents_filtered.append(('MICROBE_NAME', name, s, e))
                        
            elif shape == 'Xxxx xxxx':
                resolved_basionym = resolve_basionym(name)
                name_split = name.split(' ')
                shortform_name = name_split[0][0] + '. ' + name_split[1]
                #then add it to 'resolved'
                mic_ents_resolved.append(('MICROBE_NAME', resolved_basionym, s, e))
                mic_ents_filtered.append(('MICROBE_NAME', name, s, e))

        chem_ents_resolved = []
        
        for sci in all_ents_raw.ents:
            sci_text = sci.text.lower()
            if 'CHEMICAL' in sci.label_ and sci_text not in not_chemicals + [text.lower() for l, text, s, e in mic_ents_filtered]:
                """ if noresolve_chemicals_mapping.get(sci_text):
                    chem_ents_resolved.append(('CHEMICAL', noresolve_chemicals_mapping.get(sci_text), sci.start_char, sci.end_char))
                    continue """
                
                """ if fixed_chemical_resolutions.get(sci_text):
                    print("resolved:", sci_text, "to", fixed_chemical_resolutions[sci_text])
                    chem_ents_resolved.append(('CHEMICAL', fixed_chemical_resolutions[sci_text], sci.start_char, sci.end_char))
                    continue """

                chem_ents_resolved.append(('CHEMICAL', sci_text, sci.start_char, sci.end_char))

                """ if chemical_mapping.get(sci_text):
                    chem_ents_resolved.append(('CHEMICAL', chemical_mapping[sci_text], sci.start_char, sci.end_char))
                else:
                    comps = pcp.get_compounds(sci_text, 'name')
                    time.sleep(0.2)
                    if comps:
                        if not chemical_mapping.get(sci_text):
                            if comps[0].synonyms:
                                chemical_mapping[sci_text] = comps[0].synonyms[0].lower()
                                chem_ents_resolved.append(('CHEMICAL', comps[0].synonyms[0].lower(), sci.start_char, sci.end_char))
                            else:
                                chemical_mapping[sci_text] = sci_text
                                chem_ents_resolved.append(('CHEMICAL', sci_text, sci.start_char, sci.end_char)) """
                        
                    #else do nothing since if there are no results then it must not be a chemical

        """ utils_found = re.finditer('|'.join(utilizations), str(sent).lower())
        utilization_ents = [('UTILIZATION', sent[util.start(0):util.end(0)].lower(), util.start(0), util.end(0)) for util in utils_found] """
        
        ents_and_labels_resolved = [(label, text, start, end) for label, text, start, end in mic_ents_resolved + chem_ents_resolved if label in custom_labels and text not in not_chemicals and len(text)>=3]
        ents_and_labels_resolved = sorted(ents_and_labels_resolved, key= lambda tup: tup[2])
            
        if not merge_sentence:
            #if there are entities which are not empty
            if ents_and_labels_resolved:
                merge_sentence = sent
                merge_ents_resolved = ents_and_labels_resolved
        #but if there is a sentence awaiting merging
        else:
            #if there are any entities to merge with, check that they arent all 'UTILIZATION'
            if ents_and_labels_resolved:
                """ labels = [label for label, t,s,e in ents_and_labels_resolved]
                if not (len(set(labels)) == 1): """
                ents_and_labels_add = [(l, t, s+len(merge_sentence)+1, e+len(merge_sentence)+1) for l, t, s, e in ents_and_labels_resolved]
                merge_ents_resolved = merge_ents_resolved + ents_and_labels_add
                merge_sentence = merge_sentence + ' ' + sent
                    
        all_merged_labels_resolved = [label for label,t,s,e in merge_ents_resolved]
      
        #checking to see if the merged ents can be written to the file, or if we should just keep merging until theres enough to form a 'relation'
        #also have to make sure there isnt only one chemical if the other 2 are utilisations
        if len(merge_ents_resolved) >= 3:
            microbes = list(set([mic for label,mic,s,e in merge_ents_resolved if 'MICROBE' in label]))
            no_microbes = len(microbes)
            chemicals = list(set([text for label, text,s,e in merge_ents_resolved if 'CHEMICAL' in label])) #the types of chemicals identified
            no_chemicals = len(chemicals)

            condition1 = no_microbes >= 1 and no_chemicals >= 1 #MICROBE CONSUMES/PRODUCES CHEMICAL
            #TODO: add this back in
            #condition2 = no_chemicals >= 2 and len(set(chemicals)) > 1 #CHEMICAL BECOMES CHEMICAL
            if condition1:

                """ with open('list_files\\all_paper_entities.jsonl', 'a', encoding='utf-8') as f:
                    json.dump((merge_sentence, writable_ents), f)
                    f.write('\n') """
                    
                def process_comp(comp):
                    replace_chars = [' ', '.', '?', '!', ',', '\\', '/']
                    for char in replace_chars:
                        comp = comp.replace(char, '')
                    return comp
                
                print(microbes, chemicals)

                for microbe in microbes:
                    for chem in chemicals:
                        #bacdive stuff if biocyc doesnt work out
                        """ #testing for utilisations
                        def test_metabolites(method, relation):
                            synonyms = pcp_mappings[method].get(chem)
                            print(synonyms)
                            if not synonyms:
                                return
                            metabolite_found = False
                            for syn in synonyms:
                                if syn in metabolite_utilisation.get(microbe):
                                    metabolite_found = True
                                    break
                            if metabolite_found:
                                print("relation found!")
                                new_relations[('MICROBE', microbe, relation, 'CHEMICAL', chem)] = True

                        test_metabolites('utilisation', 'CONSUMES')
                        test_metabolites('production', 'PRODUCES') """

                        dbs = biocyc_db_mappings.get(microbe)

                        if not dbs:
                            continue

                        if len(dbs) == 1:
                            db_values = dbs.values()
                            database = list(db_values)[0]
                        else:
                            db_values = dbs.values()
                            database = random.choice(list(db_values))

                        #TODO: check the biocyc common name mapping

                        if biocyc_common_name_mapping.get(chem):
                            chem_id, chem_common_name = biocyc_common_name_mapping[chem]
                            chem_common_name = process_html(chem_common_name)
                            #skip this microbe and chem combo
                            if ('MICROBE', microbe, 'CONSUMES', 'CHEMICAL', chem_common_name, True) in saved_relations or ('MICROBE', microbe, 'PRODUCES', 'CHEMICAL', chem_common_name, True) in saved_relations:
                                print("seen relation")
                                continue

                        else:
                            search_chem = process_comp(chem)
                            #if nothing comes up in compounds search assume its an enzyme which was misidentified as a chemical
                            chem_resp = session.get(f'https://websvc.biocyc.org/{database}/name-search?object={search_chem}&class=Compounds&fmt=json')

                            time.sleep(1)

                            try:
                                chem_resp = chem_resp.json()
                            except:
                                print("json error:")
                                print(chem_resp.text)
                                continue

                            if not chem_resp.get('RESULTS'):
                                #nothing to see here!
                                continue

                            chem_resp = chem_resp.get('RESULTS')[0]

                            chem_id = chem_resp.get('OBJECT-ID')
                            #TODO: use this common name in the relations instead of 'chem'
                            chem_common_name = chem_resp.get('COMMON-NAME')
                            chem_common_name = process_html(chem_common_name)

                            biocyc_common_name_mapping[chem] = (chem_id, chem_common_name)

                        utilisation = session.get(f'https://websvc.biocyc.org/getxml?{database}:{chem_id}&fmt=json')

                        #get synonyms and add them to the common names map? maybe?

                        time.sleep(1)

                        try:
                            util_xml = ET.fromstring(utilisation.text).find('Compound')
                        except: #accept defeat and move on
                            continue

                        lefts = util_xml.find('appears-in-left-side-of')

                        if not lefts == None:
                            print("new relation!")
                            #add a relation that the microbe consumes the chemical as a substrate
                            saved_relations.add(('MICROBE', microbe, 'CONSUMES', 'CHEMICAL', chem_common_name, True))

                            for reaction in lefts:
                                reaction_id = reaction.get('frameid')

                                """ if reaction_id in biocyc_saved_ids:
                                    continue

                                biocyc_saved_ids.append(reaction_id) """

                                if reaction_enzyme_mapping.get(reaction_id):
                                    for enzyme in reaction_enzyme_mapping[reaction_id]:
                                        #when adding the 'consumes' relation perhaps add the enzyme in the metadata?
                                        saved_relations.add(('MICROBE', microbe, 'SYNTHASIZES', 'ENZYME', enzyme, True))
                                        saved_relations.add(('ENZYME', enzyme, 'CATALYZES', 'CHEMICAL', chem_common_name, True))
                                    continue

                                enzyme_names = []

                                enzymes_resp = session.get(f"https://websvc.biocyc.org/apixml?fn=enzymes-of-reaction&id={database}:{reaction_id}")

                                time.sleep(1)

                                try:
                                    enzymes_xml = ET.fromstring(enzymes_resp.text)
                                except:
                                    print("enzyme failure:")
                                    reaction_enzyme_mapping[reaction_id] = []
                                    continue

                                proteins = enzymes_xml.findall('Protein')

                                for protein in proteins:

                                    protein_common_name = protein.find('common-name').text
                                    protein_common_name = process_html(protein_common_name)
                                    protein_id = protein.get('frameid')

                                    enzyme_names.append(protein_common_name)

                                    entity_label_map[protein_id] = preprocess_name(protein_common_name)

                                    saved_relations.add(('MICROBE', microbe, 'SYNTHASIZES', 'ENZYME', protein_common_name, True))
                                    saved_relations.add(('ENZYME', protein_common_name, 'CATALYZES', 'CHEMICAL', chem_common_name, True))

                                    time.sleep(1)

                                reaction_enzyme_mapping[reaction_id] = enzyme_names

                                #print(enzymes_xml.findall('enzyme'))
                        else:
                            saved_relations.add(('MICROBE', microbe, 'CONSUMES', 'CHEMICAL', chem_common_name, False))
                                
                        try:
                            rights = util_xml.find('appears-in-right-side-of')
                        except:
                            rights = None

                        if not rights == None:
                            #add a relation that the microbe produces this metabolite
                            print("produced relation!")
                            saved_relations.add(('MICROBE', microbe, 'PRODUCES', 'CHEMICAL', chem_common_name, True))
                        else:
                            saved_relations.add(('MICROBE', microbe, 'PRODUCES', 'CHEMICAL', chem_common_name, False))
                        #dont add a false because of absence, unless a false could indicate lack of information
                        #the falses will be ignored in cypher command generation, but in the future this could be flagged up as a means to find another method of relation extraction

                        """ for right in rights:
                            print(right.get('frameid')) """
                    
                merge_sentence = ''
                merge_ents = []
                merge_ents_resolved = []
                last_sent_length = 0

                with open("list_files\\entity_label_map.json", 'w') as f:
                    json.dump(entity_label_map, f)

                #TODO: save the new relations as well
                #this would involve converting the relation into a string format for serialisation... which would then have to be converted back to form the cypher command
                #this is confusing
                #alternative idea: instead of storing maps, just store everything as tuples in a list, so that we dont need to use tuples as keys in a dict, yeah!

                with open('list_files\\saved_relations.json', 'w', encoding='utf-8') as f:
                    json.dump(list(saved_relations), f)

                with open('list_files\\reaction_enzyme_mapping.json', 'w', encoding='utf-8') as f:
                    json.dump(reaction_enzyme_mapping, f)

                with open('list_files\\pcp_chemical_mapping.json', 'w', encoding='utf-8') as f:
                    json.dump(chemical_mapping, f)

                with open('list_files\\biocyc_common_name_mapping.json', 'w', encoding='utf-8') as f:
                    json.dump(biocyc_common_name_mapping, f)
                
    del sentences

with open('list_files\\paper_number.json', 'w') as f:
    json.dump(paper_no+3, f)

0
['Gluconobacter oxydans']
new relation!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relation!
new relation!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relation!
new relation!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
new relations!
ne

KeyboardInterrupt: 

Create cypher commands

In [ ]:
with open('list_files\\saved_relations.json', 'r', encoding='utf-8') as f:
    saved_relations = json.load(f)

for ent1type, ent1, rel, ent2type, ent2, value in saved_relations:
    if not value:
        continue
    ent1name = preprocess_name(ent1)
    ent1label = preprocess_label(ent1name)

    ent2name = preprocess_name(ent2)
    ent2label = preprocess_label(ent2name)

    if not entity_label_map.get(ent1label):
        ent1string = f"MERGE({ent1label}:{ent1type}" + r"{" + f"name: '{ent1name}" + r"'})"
        with open('all_relations.cypher', 'a', encoding='utf-8') as f:
            f.write(ent1string)
            f.write('\n')

        entity_label_map[ent1label] = ent1name

    if not entity_label_map.get(ent2label):
        ent2string = f"MERGE({ent2label}:{ent2type}" + r"{" + f"name: '{ent2name}" + r"'})"
        with open('all_relations.cypher', 'a', encoding='utf-8') as f:
            f.write(ent2string)
            f.write('\n')

        entity_label_map[ent2label] = ent2name

    relstring = f"MERGE ({ent1label})-[:{rel}]->({ent2label})"

    with open('all_relations.cypher', 'a', encoding='utf-8') as f:
        f.write(relstring)
        f.write('\n')

reset

In [10]:
#%%script False
import json

with open("list_files\\entity_label_map.json", 'w') as f:
    json.dump({}, f)

with open('list_files\\paper_number.json', 'w') as f:
    json.dump(0, f)

with open('list_files\\saved_relations.json', 'w', encoding='utf-8') as f:
    json.dump([], f)

In [ ]:
%%script False

""" sci_lents = linker(sci_ents)
        linker_labels = []
        for ent in sci_ents.ents:
            #print([linker.kb.cui_to_entity[lent[0]] for lent in ent._.kb_ents])
            if 'CHEMICAL' in ent.label_ and ent.text.lower() not in not_chemicals + [ent.text.lower() for l, text, s, e in mic_ents]:
                if ent._.kb_ents:
                    #print(ent._.kb_ents[0])
                    label = linker.kb.cui_to_entity[ent._.kb_ents[0][0]]
                    linker_labels.append(label.canonical_name)
                else:
                    linker_labels.append(ent.text) 
                    
        linked_sci_ents = [lent for lent in sci_lents.ents if 'CHEMICAL' in lent.label_] """